In [ ]:
import numpy as np
import pandas as pd
import zipfile

with zipfile.ZipFile('future_text.zip') as zip_ref:
    zip_ref.extractall()

train_df = pd.read_csv('train_data.csv')
test_df = pd.read_csv('test_data.csv')
train_df.head(3)

,english,romanian
0,"If you add three and four, you get seven.","Dacă aduni trei cu patru, obții șapte."
1,"At first, I didn't like it, but it gradually b...","La început nu mi-a plăcut, dar treptat a deven..."
2,He made me love jazz.,El m-a făcut să îndrăgesc jazz-ul.


In [2]:
test_df.head(3)

,datapointID,Word,Similar
0,1,beverage,pasionat
1,2,analyze,adopta
2,3,film,audio


# Subtask 1

In [3]:
en_words_s1 = set(test_df['Word'].str.lower())
ro_words_s1 = set(test_df['Similar'].str.lower())
answer_s1 = len(en_words_s1 & ro_words_s1)

answer_s1

124

# Subtask 2

In [4]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models import KeyedVectors
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

en_embbeder = KeyedVectors.load_word2vec_format('wiki.en.small.vec', binary=False)
ro_embbeder = KeyedVectors.load_word2vec_format('wiki.ro.small.vec', binary=False)

def get_sentence_embedding(sentence, model, stop_words):
    tokens = word_tokenize(sentence.lower())
    vectors = [model[w] for w in tokens if w in model and w not in stop_words and w.isalnum()]

    if not vectors:
        return None
    return np.mean(vectors, axis=0)

In [5]:
X_train = []  #english
y_train = []  #romanian

train_df = train_df.dropna()

for _, row in train_df.iterrows():
    en_vec = get_sentence_embedding(row['english'], en_embbeder, stopwords.words('english'))
    ro_vec = get_sentence_embedding(row['romanian'], ro_embbeder, stopwords.words('romanian'))

    if en_vec is not None and ro_vec is not None:
        X_train.append(en_vec)
        y_train.append(ro_vec)

X_train = np.array(X_train)
y_train = np.array(y_train)

In [6]:
from sklearn.linear_model import Ridge

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

y_test = []  #romanian words to match

for word in test_df['Similar'].tolist():
    word_lower = str(word).lower()
    y_test.append(ro_embbeder[word_lower])

y_test = np.array(y_test)
indexes = list(test_df.index)

In [11]:
#For every word in english we find the best match in romanian
from sklearn.metrics.pairwise import cosine_similarity

preds_s2 = []

for word in test_df['Word'].tolist():
    en_vec = en_embbeder[str(word).lower()].reshape(1, -1)
    pred = model.predict(en_vec)
    
    sims = cosine_similarity(pred, y_test)[0]
    best_idx = np.argmax(sims)
    best_word = test_df.iloc[best_idx]['Similar']
    preds_s2.append(best_word)

In [12]:
output_df_s1 = pd.DataFrame({
    'subtaskID': [1],
    'datapointID': [0],
    'answer': [answer_s1]
})

output_df_s2 = pd.DataFrame({
    'subtaskID': 2,
    'datapointID': test_df.index + 1,
    'answer': preds_s2
})

output_df = pd.concat([output_df_s1, output_df_s2], ignore_index=True)
output_df.to_csv('submission.csv', index=False)
output_df.head()

,subtaskID,datapointID,answer
0,1,0,124
1,2,1,băutură
2,2,2,analiza
3,2,3,film
4,2,4,cupolă
